In [ ]:
from pathlib import Path
import os
import json
import joblib
import numpy as np
import yaml

from dotenv import load_dotenv
from groq import Groq
from pydantic import BaseModel
from typing import List

from src.schemas.support_schema import SupportAnalysis
from src.nlp.preprocessing import clean_text
from src.llm.llm_analyzer import analyze_with_llm

In [ ]:
class SupportAnalysis(BaseModel):
    intent: str
    sentiment: str
    urgency: str
    summary: str
    entities: List[str]
    root_cause: str
    recommended_action: str
    department: str
    escalation_required: bool
    customer_response: str

In [ ]:
load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

API key loaded: True


In [ ]:
client = Groq(api_key=api_key)

Groq client initialized successfully.


In [ ]:
with open("../config.yaml", "r") as file:
    config = yaml.safe_load(file)


{'llm': {'provider': 'groq', 'model': 'openai/gpt-oss-20b', 'temperature': 0.2}}


In [ ]:
llm_config = config["llm"]

model_name = llm_config["model"]
temperature = llm_config["temperature"]

Provider: groq
Model: openai/gpt-oss-20b
Temperature: 0.2


In [ ]:
prompt_path = Path("../prompts/support_analysis.txt")

prompt_template = prompt_path.read_text(encoding="utf-8")

You are an AI customer support intelligence assistant for a banking support system.

Analyze the customer's support query and return structured information about the issue.

Your analysis must identify:

1. The customer's intent.
2. The customer's sentiment.
3. The urgency of the issue.
4. A concise summary of the customer's problem.
5. Important entities mentioned in the query.
6. A plausible root-cause hypothesis.
7. A recommended action for the support team.
8. The appropriate department for handling the issue.
9. Whether the issue requires escalation.
10. A concise and professional response that can be sent to the customer.

Customer Query:
{customer_query}


In [ ]:
customer_query = "I can't verify my identity."

prompt = prompt_template.format(
    customer_query=customer_query
)

You are an AI customer support intelligence assistant for a banking support system.

Analyze the customer's support query and return structured information about the issue.

Your analysis must identify:

1. The customer's intent.
2. The customer's sentiment.
3. The urgency of the issue.
4. A concise summary of the customer's problem.
5. Important entities mentioned in the query.
6. A plausible root-cause hypothesis.
7. A recommended action for the support team.
8. The appropriate department for handling the issue.
9. Whether the issue requires escalation.
10. A concise and professional response that can be sent to the customer.

Customer Query:
I can't verify my identity.


In [ ]:
response = client.chat.completions.create(
    model=model_name,
    temperature=temperature,
    messages=[
        {
            "role": "system",
            "content": """You are SupportIQ, an intelligent banking customer-support analysis system.

You MUST return ONLY valid JSON.
Do not use Markdown.
Do not include explanations outside the JSON.

The JSON must contain exactly these fields:
- intent
- sentiment
- urgency
- summary
- entities
- root_cause
- recommended_action
- department
- escalation_required
- customer_response
"""
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    response_format={"type": "json_object"}
)

llm_output = response.choices[0].message.content


{"intent":"identity_verification_help","sentiment":"negative","urgency":"moderate","summary":"Customer is unable to complete identity verification process","entities":["identity verification"],"root_cause":"Technical issue with verification portal or missing/incorrect documents","recommended_action":"Ask customer to provide details of the error, verify required documents, guide them through the correct steps, and offer to reset the verification process if needed","department":"Identity & Access","escalation_required":"false","customer_response":"I’m sorry you’re having trouble verifying your identity. Let’s get this sorted out right away. Could you please let me know what error message you’re seeing or which step you’re stuck on? I’ll guide you through the correct process and, if needed, reset your verification attempt so you can access your account. Thank you for your patience."}


In [ ]:
parsed_output = json.loads(llm_output)

{'intent': 'identity_verification_help', 'sentiment': 'negative', 'urgency': 'moderate', 'summary': 'Customer is unable to complete identity verification process', 'entities': ['identity verification'], 'root_cause': 'Technical issue with verification portal or missing/incorrect documents', 'recommended_action': 'Ask customer to provide details of the error, verify required documents, guide them through the correct steps, and offer to reset the verification process if needed', 'department': 'Identity & Access', 'escalation_required': 'false', 'customer_response': 'I’m sorry you’re having trouble verifying your identity. Let’s get this sorted out right away. Could you please let me know what error message you’re seeing or which step you’re stuck on? I’ll guide you through the correct process and, if needed, reset your verification attempt so you can access your account. Thank you for your patience.'}


In [ ]:
validated_output = SupportAnalysis(**parsed_output)

intent='identity_verification_help' sentiment='negative' urgency='moderate' summary='Customer is unable to complete identity verification process' entities=['identity verification'] root_cause='Technical issue with verification portal or missing/incorrect documents' recommended_action='Ask customer to provide details of the error, verify required documents, guide them through the correct steps, and offer to reset the verification process if needed' department='Identity & Access' escalation_required=False customer_response='I’m sorry you’re having trouble verifying your identity. Let’s get this sorted out right away. Could you please let me know what error message you’re seeing or which step you’re stuck on? I’ll guide you through the correct process and, if needed, reset your verification attempt so you can access your account. Thank you for your patience.'


Implementing ML with LLM

In [22]:
svm_model = joblib.load("../models/svm_model.pkl")
tfidf_vectorizer = joblib.load("../models/tfidf_vectorizer.pkl")

In [23]:
def predict_intent(text):
    clean = clean_text(text)

    vector = tfidf_vectorizer.transform([clean])

    prediction = svm_model.predict(vector)[0]

    scores = svm_model.decision_function(vector)[0]

    sorted_scores = np.sort(scores)[::-1]

    margin = sorted_scores[0] - sorted_scores[1]

    return prediction, margin

In [ ]:
CONFIDENCE_THRESHOLD = 0.5

def analyze_customer_query(customer_query):
    intent, margin = predict_intent(customer_query)

    if margin >= CONFIDENCE_THRESHOLD:
        return {
            "route": "ML",
            "intent": intent,
            "margin": float(margin),
            "llm_analysis": None
        }

    llm_result = analyze_with_llm(customer_query)

    return {
        "route": "LLM",
        "intent": llm_result.intent,
        "margin": float(margin),
        "llm_analysis": llm_result
    }

In [ ]:
test_query = "I can't verify my identity."

result = analyze_customer_query(test_query)

print("Customer Query:", test_query)
print("Route:", result["route"])
print("Intent:", result["intent"])
print("Margin:", result["margin"])

Customer Query: I can't verify my identity.
Route: ML
Intent: verify_my_identity
Margin: 0.8897278108143605


In [ ]:
test_query = "Can I be topped up once I hit a certain balance?"

result = analyze_customer_query(test_query)

print("Customer Query:", test_query)
print("Route:", result["route"])
print("Intent:", result["intent"])
print("Margin:", result["margin"])

if result["llm_analysis"]:
    print("\nLLM Analysis:")
    print("Intent:", result["llm_analysis"].intent)
    print("Sentiment:", result["llm_analysis"].sentiment)
    print("Urgency:", result["llm_analysis"].urgency)
    print("Summary:", result["llm_analysis"].summary)

Customer Query: Can I be topped up once I hit a certain balance?
Route: LLM
Intent: inquire_auto_top_up
Margin: 0.0002402639592914424

LLM Analysis:
Intent: inquire_auto_top_up
Sentiment: neutral
Urgency: low
Summary: User asks if they can automatically top up their account when balance reaches a certain threshold.
